<a href="https://colab.research.google.com/github/GuFerreiraV/notebooks-google-colab/blob/main/automatizando_commits_com_ia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Automação com IA
- Script em python que automatiza commits usando o modelo do llama 3
- Não sei se esse script funciona no linux, irei testar posteriormente no `crontab`
- Por enquanto estou usando em um repositório de anotações privadas.
- Alguns problemas que notei:
  - No Windows, toda vez que ele vai fazer o push, abre uma janela pop-up do git perguntando que o usuário está fazendo o push, isso pode ser chato.

In [ ]:
pip install watchdog google-generativeai groq

In [ ]:
import time
import subprocess
import os
from groq import Groq
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

# --- CONFIGURAÇÃO ---
client = Groq(api_key="SUA_API_KEY")

# Caminho do seu Obsidian (usando 'r' para evitar erro de escape no Windows)
PATH_TO_WATCH = r"caminho_do_seu_vault"

IGNORED_DIRS = {'.obsidian', '.git', '.trash', 'bin', 'obj'}
IGNORED_EXTENSIONS = {'.json', '.sync', '.tmp', '.py'} # Ignora o próprio script

class ObsidianGitHandler(FileSystemEventHandler):
    def on_modified(self, event):
        path_parts = os.path.normpath(event.src_path).split(os.sep)

        if event.is_directory: return

        if any(ignored in path_parts for ignored in IGNORED_DIRS): return
        if os.path.splitext(event.src_path)[1] in IGNORED_EXTENSIONS: return

        print(f"🔍 Evento detectado: {event.event_type} em {event.src_path}", flush=True)

        print(f"Alteração: {event.src_path}")
        time.sleep(15)
        self.sync_obsidian()

    def sync_obsidian(self):
        try:
            # Garante que estamos no diretório correto para o Git
            os.chdir(PATH_TO_WATCH)

            subprocess.run(["git", "add", "."], check=True)
            diff = subprocess.check_output(["git", "diff", "--cached"]).decode("utf-8")

            if not diff or len(diff.strip()) < 5: return

            prompt = f"""
            Analise as alterações nas notas Markdown abaixo.
            Gere uma mensagem de commit curta em português: <tag>: <descrição>

            Tags para Obsidian:
            - docs: Nova nota ou conteúdo novo.
            - refactor: Reorganização de texto ou mudança de links/pastas.
            - style: Mudança de formatação (negrito, listas, cores).
            - chore: Atualização de metadados ou propriedades (YAML).
            - fix: Correção de erros ortográficos ou links quebrados.

            Diff:
            {diff[:1500]}
            """

            chat_completion = client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama-3.3-70b-versatile", # modelo
            )
            commit_msg = chat_completion.choices[0].message.content.strip()

            print(f"Enviando: {commit_msg}")
            subprocess.run(["git", "commit", "-m", commit_msg], check=True)
            subprocess.run(["git", "pull", "--rebase"], check=True)
            subprocess.run(["git", "push"], check=True)
            print("✅ Sincronizado com sucesso!")

        except Exception as e:
            print(f"Aguardando (Erro: {e})")

if __name__ == "__main__":
    event_handler = ObsidianGitHandler()
    observer = Observer()
    observer.schedule(event_handler, PATH_TO_WATCH, recursive=True)
    observer.start()
    print(f"Monitorando: {PATH_TO_WATCH}", flush=True)
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        observer.stop()
    observer.join()